In [1]:
import requests
import pandas as pd

print('Libraries imported')
print(f'Pandas: {pd.__version__}')

Libraries imported
Pandas: 3.0.3


In [2]:
# paste your API key from openweathermap.org
API_KEY = "a90ece615d6beed52df4e2e676335810"

BASE_URL = "https://api.openweathermap.org/data/2.5/weather"

CITIES = ['Mumbai', 'Delhi', 'Bangalore', 'Chennai',
          'Hyderabad', 'Kolkata', 'Pune', 'Jaipur',
          'Ahmedabad', 'Lucknow']

print(f'Cities to fetch: {CITIES}')

Cities to fetch: ['Mumbai', 'Delhi', 'Bangalore', 'Chennai', 'Hyderabad', 'Kolkata', 'Pune', 'Jaipur', 'Ahmedabad', 'Lucknow']


In [3]:
# EXTRACT — fetch weather for each city

weather_records = []

for city in CITIES:
    params = {'q': city, 'appid': API_KEY, 'units': 'metric'}
    response = requests.get(BASE_URL, params=params, timeout=10)

    if response.status_code == 200:
        data = response.json()
        weather_records.append({
            'city':        data['name'],
            'temperature': data['main']['temp'],
            'feels_like':  data['main']['feels_like'],
            'humidity':    data['main']['humidity'],
            'pressure':    data['main']['pressure'],
            'wind_speed':  data['wind']['speed'],
            'condition':   data['weather'][0]['description'],
            'visibility':  data.get('visibility', 0) // 1000  # metres to km
        })
        print(f'{city} -> {data["main"]["temp"]}°C, {data["weather"][0]["description"]}')
    else:
        print(f'{city} -> FAILED ({response.status_code})')

print(f'\nFetched: {len(weather_records)}/{len(CITIES)} cities')

Mumbai -> 31.99°C, haze
Delhi -> 30.05°C, haze
Bangalore -> 25.51°C, overcast clouds
Chennai -> 34.77°C, few clouds
Hyderabad -> 31.23°C, broken clouds
Kolkata -> 32.97°C, haze
Pune -> 29.67°C, clear sky
Jaipur -> 29.1°C, clear sky
Ahmedabad -> 31.02°C, haze
Lucknow -> 27.99°C, haze

Fetched: 10/10 cities


In [4]:
# fallback data — run this only if API key is not working

if len(weather_records) == 0:
    print('API not available, using fallback data')
    weather_records = [
        {'city': 'Mumbai',    'temperature': 32.5, 'feels_like': 36.0, 'humidity': 78, 'pressure': 1009, 'wind_speed': 5.2, 'condition': 'partly cloudy', 'visibility': 8},
        {'city': 'Delhi',     'temperature': 39.0, 'feels_like': 42.0, 'humidity': 20, 'pressure': 998,  'wind_speed': 4.1, 'condition': 'haze',          'visibility': 5},
        {'city': 'Bangalore', 'temperature': 26.0, 'feels_like': 27.0, 'humidity': 72, 'pressure': 1013, 'wind_speed': 3.5, 'condition': 'cloudy',        'visibility': 10},
        {'city': 'Chennai',   'temperature': 33.0, 'feels_like': 37.0, 'humidity': 68, 'pressure': 1008, 'wind_speed': 4.8, 'condition': 'haze',          'visibility': 6},
        {'city': 'Hyderabad', 'temperature': 30.5, 'feels_like': 33.0, 'humidity': 55, 'pressure': 1010, 'wind_speed': 3.9, 'condition': 'haze',          'visibility': 7},
        {'city': 'Kolkata',   'temperature': 27.0, 'feels_like': 30.0, 'humidity': 85, 'pressure': 1007, 'wind_speed': 2.8, 'condition': 'thunderstorm',  'visibility': 4},
        {'city': 'Pune',      'temperature': 29.3, 'feels_like': 31.0, 'humidity': 55, 'pressure': 1014, 'wind_speed': 3.1, 'condition': 'partly cloudy', 'visibility': 9},
        {'city': 'Jaipur',    'temperature': 40.1, 'feels_like': 43.0, 'humidity': 22, 'pressure': 998,  'wind_speed': 5.5, 'condition': 'sunny',         'visibility': 12},
        {'city': 'Ahmedabad', 'temperature': 33.0, 'feels_like': 36.0, 'humidity': 55, 'pressure': 1005, 'wind_speed': 4.0, 'condition': 'smoke',         'visibility': 6},
        {'city': 'Lucknow',   'temperature': 34.0, 'feels_like': 37.0, 'humidity': 46, 'pressure': 1002, 'wind_speed': 3.7, 'condition': 'haze',          'visibility': 5},
    ]
    print(f'Fallback data loaded for {len(weather_records)} cities')
else:
    print(f'Using live data for {len(weather_records)} cities')

Using live data for 10 cities


In [5]:
# create raw dataframe
df_raw = pd.DataFrame(weather_records)

print(f'Shape: {df_raw.shape}')
print(df_raw)

Shape: (10, 8)
        city  temperature  feels_like  humidity  pressure  wind_speed  \
0     Mumbai        31.99       38.97        66      1011        3.09   
1      Delhi        30.05       31.79        54      1008        4.12   
2  Bengaluru        25.51       26.07        75      1014       10.73   
3    Chennai        34.77       41.72        54      1009        5.66   
4  Hyderabad        31.23       32.04        45      1010        5.66   
5    Kolkata        32.97       39.97        70      1008        2.57   
6       Pune        29.67       29.54        42      1012        3.15   
7     Jaipur        29.10       29.21        45      1008        3.32   
8  Ahmedabad        31.02       33.70        55      1009        2.06   
9    Lucknow        27.99       30.52        69      1008        3.09   

         condition  visibility  
0             haze           5  
1             haze           4  
2  overcast clouds          10  
3       few clouds           6  
4    broken clou

In [6]:
# check data quality
print('Missing values:')
print(df_raw.isnull().sum())
print(f'\nDuplicates: {df_raw.duplicated().sum()}')
print(f'\nData types:')
print(df_raw.dtypes)

Missing values:
city           0
temperature    0
feels_like     0
humidity       0
pressure       0
wind_speed     0
condition      0
visibility     0
dtype: int64

Duplicates: 0

Data types:
city               str
temperature    float64
feels_like     float64
humidity         int64
pressure         int64
wind_speed     float64
condition          str
visibility       int64
dtype: object


In [7]:
# TRANSFORM — clean and add new columns
df = df_raw.copy()

# fix casing
df['city']      = df['city'].str.strip().str.title()
df['condition'] = df['condition'].str.strip().str.title()

# round numbers
df['temperature'] = df['temperature'].round(1)
df['feels_like']  = df['feels_like'].round(1)

# heat category
def heat_category(temp):
    if temp >= 40:   return 'Extreme Heat'
    elif temp >= 35: return 'Very Hot'
    elif temp >= 30: return 'Hot'
    elif temp >= 25: return 'Warm'
    else:            return 'Comfortable'

df['heat_category'] = df['temperature'].apply(heat_category)

# humidity level
def humidity_level(h):
    if h >= 70:   return 'High'
    elif h >= 40: return 'Moderate'
    else:         return 'Low'

df['humidity_level'] = df['humidity'].apply(humidity_level)

# how much hotter it feels vs actual temp
df['feels_diff'] = (df['feels_like'] - df['temperature']).round(1)

print('Transform done')
print(df[['city', 'temperature', 'heat_category', 'humidity', 'humidity_level', 'condition']])

Transform done
        city  temperature heat_category  humidity humidity_level  \
0     Mumbai         32.0           Hot        66       Moderate   
1      Delhi         30.0           Hot        54       Moderate   
2  Bengaluru         25.5          Warm        75           High   
3    Chennai         34.8           Hot        54       Moderate   
4  Hyderabad         31.2           Hot        45       Moderate   
5    Kolkata         33.0           Hot        70           High   
6       Pune         29.7          Warm        42       Moderate   
7     Jaipur         29.1          Warm        45       Moderate   
8  Ahmedabad         31.0           Hot        55       Moderate   
9    Lucknow         28.0          Warm        69       Moderate   

         condition  
0             Haze  
1             Haze  
2  Overcast Clouds  
3       Few Clouds  
4    Broken Clouds  
5             Haze  
6        Clear Sky  
7        Clear Sky  
8             Haze  
9             Haze  


In [8]:
# analysis
print('=== Weather Analysis ===')

hottest   = df.loc[df['temperature'].idxmax()]
coolest   = df.loc[df['temperature'].idxmin()]
most_humid = df.loc[df['humidity'].idxmax()]

print(f'Hottest city    : {hottest["city"]} ({hottest["temperature"]}°C)')
print(f'Coolest city    : {coolest["city"]} ({coolest["temperature"]}°C)')
print(f'Most humid city : {most_humid["city"]} ({most_humid["humidity"]}%)')

print(f'\nAvg temperature : {df["temperature"].mean():.1f}°C')
print(f'Avg humidity    : {df["humidity"].mean():.1f}%')
print(f'Avg wind speed  : {df["wind_speed"].mean():.1f} m/s')

print('\nHeat categories:')
print(df['heat_category'].value_counts())

print('\nHumidity levels:')
print(df['humidity_level'].value_counts())

=== Weather Analysis ===
Hottest city    : Chennai (34.8°C)
Coolest city    : Bengaluru (25.5°C)
Most humid city : Bengaluru (75%)

Avg temperature : 30.4°C
Avg humidity    : 57.5%
Avg wind speed  : 4.3 m/s

Heat categories:
heat_category
Hot     6
Warm    4
Name: count, dtype: int64

Humidity levels:
humidity_level
Moderate    8
High        2
Name: count, dtype: int64


In [9]:
# validation
print(f'Rows          : {len(df)}')
print(f'Columns       : {len(df.columns)}')
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'Duplicates    : {df.duplicated().sum()}')
print(f'Clean         : {df.isnull().sum().sum() == 0 and df.duplicated().sum() == 0}')

Rows          : 10
Columns       : 11
Missing values: 0
Duplicates    : 0
Clean         : True


In [10]:
# LOAD — save to CSV
df.to_csv('weather_data.csv', index=False)

print('Saved to weather_data.csv')
print(f'{len(df)} rows, {len(df.columns)} columns')
print('\nETL Pipeline Complete!')
print('EXTRACT   -> fetched from OpenWeatherMap API')
print('TRANSFORM -> cleaned and added new columns')
print('LOAD      -> saved to weather_data.csv')

Saved to weather_data.csv
10 rows, 11 columns

ETL Pipeline Complete!
EXTRACT   -> fetched from OpenWeatherMap API
TRANSFORM -> cleaned and added new columns
LOAD      -> saved to weather_data.csv
